# MAE embedder

Trains a multi-channel masked autoencoder on LLC4320 cutouts and writes the trained
embedder to disk, so downstream notebooks can load it without retraining.

Preprocessing is `CutoutDataset.preprocess_for_training` -- log10 on the
grad-magnitude channels, `div_by_f` on the kinematic ones, then one global z-score
per channel.  `get_patches` defaults to `div_by_f=False`, so raw-patch baselines are
not directly comparable to this input unless they are rebuilt the same way.

In [ ]:
from pathlib import Path

import numpy as np
import torch
from einops import rearrange
from matplotlib import pyplot as plt

import llc_cutout_dataloader.cutouts_dataset as cutouts_dataset
import dl_embedding.mae as mae

## Config

In [ ]:
SOURCE = dict(bucket="dbof", folder="cutouts_dataset_v2", run_id="1_00",
              dataset_name="cutout_dataset.zarr",
              s3_endpoint="https://s3-west.nrp-nautilus.io")

DOWNLOAD = dict(subset=False, subsample_per_chunk=64, num_sample_chunks=1, n_workers=4)

CHANNELS = ["Theta", "Salt", "gradb2", "relative_vorticity", "strain_mag", "divergence", "coriolis"]

# patch_size is the MAE token size: 64x64 cutouts at 8 -> an 8x8 token grid, one
# embedding per 8x8 region, matching the DINO/raw-patch granularity.
MODEL = dict(patch_size=8, embed_dim=192, depth=6, num_heads=6,
             decoder_dim=96, decoder_depth=2, decoder_heads=3, mask_ratio=0.75)

TRAIN = dict(epochs=40, batch_size=64, lr=1.5e-4, weight_decay=0.05, seed=0)

# vorticity / strain / divergence divided by f, so a value means the same dynamics at
# every latitude; |f| is floored at its EQUATOR_DEG value so the tropics stay finite
PREP = dict(div_by_f=True, equator_deg=5.0)

VAL_FRAC = 0.2
CKPT = Path("artifacts/mae_embedder.pt")

## Load the cutouts

In [ ]:
source = cutouts_dataset.CutoutDataSource(**SOURCE)
dataset = cutouts_dataset.CutoutDataset.from_source(source=source, data_channels=CHANNELS,
                                                    **DOWNLOAD)
print(dataset.X.shape, "|", dataset.channel_names)

## Preprocess

In [ ]:
images = dataset.preprocess_for_training(**PREP)    # (N, C, H, W)
print(images.shape, "| log10 applied to:", dataset.log_scaled_channels)
print("divided by f:", [c for c in dataset.channel_names
                        if c in cutouts_dataset._DIV_SIGNED + cutouts_dataset._DIV_ABS])
print("per-channel mean", images.mean(axis=(0, 2, 3)).round(3))
print("per-channel std ", images.std(axis=(0, 2, 3)).round(3))

In [ ]:
# hold out whole cutouts, not patches, so validation loss is not scored on tokens
# whose neighbours the encoder saw during training
rng = np.random.default_rng(TRAIN["seed"])
perm = rng.permutation(len(images))
n_val = max(1, int(VAL_FRAC * len(images)))
val_idx, train_idx = perm[:n_val], perm[n_val:]
grid = images.shape[2] // MODEL["patch_size"]
print(f"train {len(train_idx)} | val {len(val_idx)} | {grid}x{grid} tokens per cutout "
      f"| {len(images) * grid * grid:,} patches total")

## Train

In [ ]:
embedder = mae.MAEEmbedder(**MODEL)
embedder.fit(images[train_idx], val_images=images[val_idx], **TRAIN)

## Save

A `state_dict` is tensors rather than a flat array set, so this is a `.pt` rather than
an `.npz`.  The checkpoint carries everything needed to rebuild the embedder: the
architecture config, the mask ratio, the channel list and cutout size that fix the
input shape, and the source/split info that identifies which data it was fitted on.

In [ ]:
def save_embedder(embedder, path, channels, img_size, **extra):
    """Write a trained MAEEmbedder plus everything needed to rebuild it."""
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({"state_dict": embedder.model.state_dict(), "cfg": embedder.cfg,
                "mask_ratio": embedder.mask_ratio, "channels": list(channels),
                "img_size": int(img_size), **extra}, path)
    return path


def load_embedder(path, device=None):
    """(MAEEmbedder, checkpoint) from save_embedder; model on device, in eval mode.

    weights_only=False because the checkpoint also carries the channel list, source
    info and split indices, not only tensors.
    """
    ck = torch.load(path, map_location="cpu", weights_only=False)
    emb = mae.MAEEmbedder(mask_ratio=ck["mask_ratio"], device=device, **ck["cfg"])
    emb.model = mae.PatchMAE(img_size=ck["img_size"], in_chans=len(ck["channels"]),
                             **ck["cfg"]).to(emb.device)
    emb.model.load_state_dict(ck["state_dict"])
    emb.model.eval()
    return emb, ck

In [ ]:
save_embedder(embedder, CKPT, dataset.channel_names, images.shape[2],
              source=dict(SOURCE), download=dict(DOWNLOAD), train=dict(TRAIN),
              prep=dict(PREP),
              source_info=dataset.source_info, ids=list(dataset.ids),
              val_index=val_idx)
print(CKPT, f"| {CKPT.stat().st_size / 1e6:.1f} MB")

In [ ]:
# the reloaded model must embed identically, or downstream notebooks are not
# working with the model that was trained here
reloaded, ck = load_embedder(CKPT)
a, b = embedder.embed(images[:8]), reloaded.embed(images[:8])
print("reload matches:", np.allclose(a, b, atol=1e-5), "| embeddings", a.shape)
print("checkpoint keys:", sorted(ck))

## Reconstructions

The reconstruction panel keeps the visible tokens and fills the masked ones with the
decoder output -- the standard MAE figure.  Agreement inside the masked blocks is the
model working; agreement outside them is the input passing through.

Colour limits come from each cutout's own 2nd-98th percentile, per channel, so the
three panels are directly comparable.

In [ ]:
def _unpatchify(x, model):
    """(B, N, p*p*C) -> (B, C, H, W); inverse of PatchMAE.patchify."""
    p, c = model.patch_size, model.in_chans
    return rearrange(x, "b (gh gw) (p1 p2 c) -> b c (gh p1) (gw p2)",
                     gh=model.grid, p1=p, p2=p, c=c)


@torch.no_grad()
def show_reconstructions(embedder, images, channel_names, n_cutouts=3, channels=None,
                         mask_ratio=None, seed=0, panel=2.6):
    """Original | masked | reconstructed, a row per channel and a figure per cutout."""
    model = embedder.model
    channel_names = list(channel_names)
    ratio = embedder.mask_ratio if mask_ratio is None else mask_ratio
    show = list(channels) if channels else channel_names
    missing = [c for c in show if c not in channel_names]
    if missing:
        raise ValueError(f"channels not in the trained set: {missing}")

    rng = np.random.default_rng(seed)
    pick = rng.choice(len(images), size=min(n_cutouts, len(images)), replace=False)
    x = torch.from_numpy(np.asarray(images[pick], dtype="float32")).to(embedder.device)

    model.eval()
    torch.manual_seed(seed)
    latent, mask, ids_restore = model.forward_encoder(x, ratio)
    pred = model.forward_decoder(latent, ids_restore)
    target = model.patchify(x)
    keep = (1.0 - mask).unsqueeze(-1)                    # 1 where the encoder saw the token
    recon = _unpatchify(target * keep + pred * (1 - keep), model).cpu().numpy()
    masked = _unpatchify(target * keep, model).cpu().numpy()
    orig = x.cpu().numpy()

    for r, i in enumerate(pick):
        fig, axes = plt.subplots(len(show), 3, squeeze=False,
                                 figsize=(panel * 3, panel * len(show)))
        for k, name in enumerate(show):
            c = channel_names.index(name)
            lo, hi = np.percentile(orig[r, c], [2, 98])
            for j, (img, tag) in enumerate(((orig[r, c], "original"),
                                            (masked[r, c], f"masked {ratio:.0%}"),
                                            (recon[r, c], "reconstructed"))):
                ax = axes[k][j]
                ax.imshow(img, cmap="viridis", vmin=lo, vmax=hi)
                ax.set_xticks([]); ax.set_yticks([])
                if k == 0:
                    ax.set_title(tag, fontsize=11)
                if j == 0:
                    ax.set_ylabel(name, fontsize=10)
        fig.suptitle(f"cutout {int(i)}", y=1.001)
        fig.tight_layout()
        plt.show()

In [ ]:
show_reconstructions(embedder, images[val_idx], dataset.channel_names, n_cutouts=3)

## Using the embedder elsewhere

```python
embedder, ck = load_embedder(Path("artifacts/mae_embedder.pt"))
F = embedder.embed(images)          # (N * grid * grid, embed_dim)
```

`embed` returns one row per token, row-major within a cutout and cutouts in dataset
order -- the same row order as `dataset.get_patches(patch_size)`, so `get_patch_coords`,
`get_patch_features` and the visualization helpers line up with it unchanged.

Feed it images built with the same transform the checkpoint records in `ck["prep"]`;
the encoder was fitted on f-normalized, z-scored inputs and will not transfer to
anything else.